# QENIVO — A100 notebook — **Round 3** (about 5–8 GPU hours)

Positive-only campaign: only experiments that passed a laptop gate and have a W10 prediction.
**Eligible (W10):** `C-BATCH-L4`, `C-PERSIST-S1`, `C-STACK-CARGO`, `C-STACK-FPRICE`.
Blocked pending gates (`crude`, `diet`, `simplex`, `qp_milp`) stay **PENDING**.
Total intended Colab budget ≈ 5–8 hours. Every part records prediction vs measured.

| Part | What it measures | Budget | W10 |
|---|---|---|---|
| stamp | GPU, driver, commit, bandwidth triad, fp64 GEMM, launch latency, L4×512 overhead; print W10 predictions | 5 min | always (all C-*) |
| crude | Crude Valuation Engine L4/L5 | 30 min | **PENDING** P-W02 |
| stack | cargo_menu / factor_price (CPU) + independent (GPU C-BATCH); omit blocked kinds | 90 min | C-BATCH-L4, C-STACK-CARGO, C-STACK-FPRICE |
| diet | W04 byte-diet ablation | 45 min | **PENDING** P-W04 |
| simplex | Stack Simplex vertices/s | 60 min | **PENDING** P-W05 |
| large_lp | Persistent vs launch (qap15 + small-S); cuPDLPx from run 1 | 45 min | C-PERSIST-S1 |
| qp_milp | W08 / W07 on VM CPU | 60 min | **PENDING** B-W07 |
| headline | `HEADLINE_ROUND3.md` from recorded rows only (prediction vs measured) | 2 min | deltas |

## How to run

1. **Runtime → Change runtime type → A100 GPU** (High-RAM if offered) → **Save**.
2. **Give it the code** (pick one):
   * set `CODE_SOURCE = "upload"` and upload **`qenivo.zip`**, or
   * set `CODE_SOURCE = "github"` to clone `https://github.com/dhruva137/Q.E.N.I.V.O.git` branch `main` (copies the `qenivo/` subfolder).
3. Upload W10 `PREDICTIONS.md` (and optionally `transfer_model_2026-10-01.json`) to
   `/content/qenivo/research_prototypes/transfer/` — if missing, eligible IDs still print
   fallbacks from the baked copy in the zip; blocked parts print **PENDING**.
4. **Runtime → Run all.** Allow multiple downloads when asked.

Crash safety: every result row is written the moment it exists; after every part a results zip is
downloaded. Re-run with the same `RUN_NAME` to resume. See `RUN_GUIDE_ROUND3.md`.

In [ ]:
# ---- settings ---------------------------------------------------------------------------------
RUN_NAME      = "round3_run1"
PARTS         = "stamp,crude,stack,diet,simplex,large_lp,qp_milp,headline"
# Hard wall per part (seconds). Notebook kills the subprocess at the cap; saved rows are kept.
BUDGET        = {
    "stamp": 5 * 60,
    "crude": 30 * 60,
    "stack": 90 * 60,
    "diet": 45 * 60,
    "simplex": 60 * 60,
    "large_lp": 45 * 60,
    "qp_milp": 60 * 60,
    "headline": 2 * 60,
}
SELF_TEST     = True       # short --quick end-to-end check before the long run
AUTO_DOWNLOAD = True
CODE_SOURCE   = "upload"   # "upload" (zip) or "github" (clone)
GITHUB_REPO   = "https://github.com/dhruva137/Q.E.N.I.V.O.git"
GITHUB_BRANCH = "main"
ZIP_NAME      = "qenivo.zip"
# Set True only when W05 Stack Simplex laptop gate passed:
import os
os.environ.setdefault("W05_PASSED", "0")

In [ ]:
# ---- setup: GPU probe, code from zip upload OR github clone, install ladder --------------------
import os, subprocess, sys, shutil, glob, json, time, zipfile
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime > Change runtime type > A100 GPU, then Run all again"
print("GPU:", gpu.stdout.strip())
RUN_DIR = f"/content/qenivo_results/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)

def project_root(folder):
    for d, dirs, fs in os.walk(folder):
        if "pyproject.toml" in fs and os.path.isdir(os.path.join(d, "src", "qenivo")):
            return d
    return None

def find_zip():
    hits = [p for p in glob.glob("/content/**/*.zip", recursive=True)
            if "qenivo_results" not in p and os.path.getsize(p) > 100_000]
    good = []
    for p in hits:
        try:
            with zipfile.ZipFile(p) as z:
                if any(n.endswith("src/qenivo/__init__.py") for n in z.namelist()):
                    good.append(p)
        except zipfile.BadZipFile:
            pass
    return sorted(good, key=lambda p: (os.path.basename(p) != ZIP_NAME, -os.path.getmtime(p)))

def install_from_root(root, label):
    shutil.rmtree("/content/qenivo", ignore_errors=True)
    shutil.copytree(root, "/content/qenivo")
    os.chdir("/content/qenivo")
    print(f"code: {label}")

src = (CODE_SOURCE or "upload").strip().lower()
if src == "github":
    dest = "/tmp/qenivo_src"
    shutil.rmtree(dest, ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GITHUB_BRANCH,
                    GITHUB_REPO, dest], check=True, timeout=600)
    cand = os.path.join(dest, "qenivo")
    root = project_root(cand) or project_root(dest)
    assert root, f"qenivo package not found under {dest} after cloning {GITHUB_BRANCH}"
    install_from_root(root, f"git {GITHUB_REPO}@{GITHUB_BRANCH}")
elif src == "upload":
    zips = find_zip()
    if not zips:
        from google.colab import files
        print(f"Choose {ZIP_NAME}:")
        files.upload()
        zips = find_zip()
    assert zips, f"Upload {ZIP_NAME} (must contain src/qenivo/__init__.py)"
    shutil.rmtree("/tmp/code_x", ignore_errors=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/tmp/code_x")
    root = project_root("/tmp/code_x")
    assert root, "zip is not a qenivo project"
    install_from_root(root, f"{zips[0]} ({os.path.getsize(zips[0])/1e6:.1f} MB)")
else:
    raise SystemExit(f"CODE_SOURCE must be 'upload' or 'github', got {CODE_SOURCE!r}")

os.environ["CUPY_CACHE_DIR"] = f"/tmp/cupy_cache_{int(time.time())}"
os.environ["PYTHONUNBUFFERED"] = "1"

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *args],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or r.stdout)[-1500:]

def importable(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

ok, err = pip("-e", ".", "highspy", "matplotlib")
if not ok:
    print("full install failed, trying step by step. pip said:", err, sep=chr(10))
    for extra in ("numpy", "scipy", "highspy", "matplotlib"):
        if not importable(extra):
            print("  ", extra, "ok" if pip(extra)[0] else "FAILED")
    ok, err = pip("--no-deps", "--no-build-isolation", "-e", ".")
    if not ok:
        ok, err = pip("--no-deps", ".")
if not importable("qenivo"):
    srcp = os.path.abspath("src")
    os.environ["PYTHONPATH"] = srcp + os.pathsep + os.environ.get("PYTHONPATH", "")
    sys.path.insert(0, srcp)
    print("using the source folder directly:", srcp)

GPU_PROBE = "from qenivo.kernels.backend import gpu_available, gpu_status; print(gpu_available(), gpu_status())"

def gpu_ok():
    r = subprocess.run([sys.executable, "-c", GPU_PROBE], capture_output=True, text=True)
    return r.stdout.strip().startswith("True"), (r.stdout.strip() or r.stderr[-300:])

if not importable("cupy"):
    pip("cupy-cuda12x")
ok_gpu, why = gpu_ok()
if not ok_gpu:
    pip("cupy-cuda12x[ctk]")
    ok_gpu, why = gpu_ok()
print("GPU kernels:", "ok" if ok_gpu else f"UNAVAILABLE, the run continues on the CPU ({why})")
chk = subprocess.run([sys.executable, "-c",
                      "import qenivo, numpy, scipy; print('qenivo', qenivo.__version__, '| numpy', numpy.__version__)"],
                     capture_output=True, text=True)
assert chk.returncode == 0, "qenivo cannot be imported: " + chk.stderr[-1500:]
print("install: ok |", chk.stdout.strip(), "| highspy", "ok" if importable("highspy") else "missing",
      "| cupy", "ok" if importable("cupy") else "missing")

# Optional: pull PREDICTIONS.md into the expected path if the owner uploaded it to /content
pred_dst = "/content/qenivo/research_prototypes/transfer/PREDICTIONS.md"
os.makedirs(os.path.dirname(pred_dst), exist_ok=True)
for cand in ("/content/PREDICTIONS.md", "/content/transfer/PREDICTIONS.md", pred_dst):
    if os.path.isfile(cand) and cand != pred_dst:
        shutil.copy2(cand, pred_dst)
        print("predictions copied from", cand)
        break
else:
    if not os.path.isfile(pred_dst):
        print("W10 PREDICTIONS.md: PENDING (upload to /content/PREDICTIONS.md to fill)")

In [ ]:
# ---- self-test, then every part with its budget; results zipped + downloaded after each part ----
import threading

def stream(cmd, budget=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    t = threading.Timer(budget, p.kill) if budget else None
    if t: t.start()
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if t: t.cancel()
    return p.returncode

def save_results(final=False):
    z = shutil.make_archive(f"/content/{RUN_NAME}_results", "zip", RUN_DIR)
    if AUTO_DOWNLOAD or final:
        try:
            from google.colab import files
            files.download(z)
        except Exception as e:
            print("download not started (results stay in", RUN_DIR, "):", e)
    return z

SCRIPT = "bench/round3_campaign.py"
remaining = sum(BUDGET.get(p.strip(), 0) for p in PARTS.split(",") if p.strip() in BUDGET)
print(f"remaining credit estimate at start: {remaining/60:.0f} min")

if SELF_TEST:
    t0 = time.time()
    shutil.rmtree("/tmp/r3_selftest", ignore_errors=True)
    rc = stream([sys.executable, "-u", SCRIPT, "--run-dir", "/tmp/r3_selftest", "--quick",
                 "--only", "stamp,crude,stack,diet,simplex,large_lp,qp_milp,headline"], 900)
    print(f"\n=== SELF-TEST {'PASSED' if rc == 0 else 'REPORTED A PROBLEM (see above); long run continues'} "
          f"in {time.time()-t0:.0f}s ===\n")

for part in PARTS.split(","):
    part = part.strip()
    if not part:
        continue
    t0 = time.time()
    rc = stream([sys.executable, "-u", SCRIPT, "--run-dir", RUN_DIR, "--only", part], BUDGET.get(part))
    hit = time.time() - t0 >= BUDGET.get(part, 1e12) - 2
    remaining = max(0, remaining - (time.time() - t0))
    print(f"--- {part}: {time.time()-t0:.0f}s"
          + ("  (budget reached: saved rows kept)" if hit else "")
          + f"  | remaining credit ~{remaining/60:.0f} min")
    save_results()

In [ ]:
# ---- the round-3 headline (from recorded rows only) ----
from IPython.display import Markdown, display
subprocess.run([sys.executable, "-u", "bench/round3_campaign.py", "--run-dir", RUN_DIR, "--only", "headline"],
               capture_output=True)
hp = f"{RUN_DIR}/HEADLINE_ROUND3.md"
if os.path.exists(hp):
    display(Markdown(open(hp).read()))
else:
    print("HEADLINE_ROUND3.md not written yet")

In [ ]:
# ---- final results zip ----
print("results zip:", save_results(final=True))